# 22 — Security, Reliability, and Maintenance

Goal: write Python that’s safe to run: avoid injection, protect secrets, and maintain systems over time.

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
python -m pip install -U cryptography
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: Threat model basics

Ask:
- What inputs are attacker-controlled?
- What resources are at risk (files, DB, network, secrets)?
- What happens if something fails?

Security is engineering + paranoia + defaults.

## 2.
L2: Command injection (subprocess)

Bad:
- `subprocess.run("ls " + user_input, shell=True)`

Better:
- pass args as a list
- avoid `shell=True`

In [ ]:

import subprocess, sys

user_input = "hello"
# safe: no shell interpolation
r = subprocess.run([sys.executable, "-c", "print('echo:', __import__('sys').argv[1])", user_input],
                   capture_output=True, text=True, check=True)
print(r.stdout.strip())


## 3.
L3: Secrets handling

Rules:
- never hardcode secrets in source
- use env vars or secret managers
- don’t log secrets
- rotate secrets

For tokens, use `secrets` module (not `random`).

In [ ]:

import os, secrets

token = secrets.token_hex(16)
os.environ["DEMO_TOKEN"] = token  # for demo only; don't set secrets like this in real code
print("token length:", len(os.environ["DEMO_TOKEN"]))


## 4.
L4: Password hashing (never store plain text)

Use a dedicated password hashing algorithm (argon2/bcrypt/scrypt).
Stdlib `hashlib` is not enough on its own for password storage.

(Shown here: *key derivation* with `hashlib.pbkdf2_hmac`, which is better than plain hash.)

In [ ]:

import os, hashlib

password = b"correct horse battery staple"
salt = os.urandom(16)
dk = hashlib.pbkdf2_hmac("sha256", password, salt, 200_000)
print("derived key bytes:", len(dk))


## 5.
L5: Deserialization risks (again)

- pickle: unsafe for untrusted inputs
- YAML: use safe loaders (e.g., `yaml.safe_load`)
- validate JSON schemas when inputs are untrusted

## 6.
L6: Dependency hygiene

- pin versions
- keep dependencies small
- update regularly
- scan for vulnerabilities (e.g., `pip-audit`, `safety`) — third-party tools

## 7.
L7: Exercises

1. Identify places in a sample app where untrusted input enters.
2. Write a helper that loads JSON and validates required keys.
3. Explain why `shell=True` is dangerous with user input.

## 8.
L8: Path traversal (filesystems)

If you accept user-provided filenames, attackers can try `../` tricks.
Mitigation:
- resolve to absolute paths
- ensure the resolved path stays within an allowed root directory

In [ ]:

from pathlib import Path

root = Path("/tmp").resolve()
user_path = Path("../etc/passwd")
resolved = (root / user_path).resolve()

print("root   :", root)
print("target :", resolved)
print("is inside root?", str(resolved).startswith(str(root)))


## 9.
L9: Safe temporary files

Use `tempfile` for temp files/dirs instead of rolling your own filenames.
Avoid predictable temp paths.